# DeepEval generation-isolation case analysis

This notebook inspects one `generator_eval.py` run. It separates pipeline/API errors from metric failures, lists every score below its recorded threshold, preserves the judge's reason, and supports detailed inspection of any case.

In [7]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

RESULTS_ROOT = next(
    path
    for path in (Path("generation_results"), Path("eval/deepEval/generation_results"))
    if path.exists()
)
RUN_ID = "2026-10-23_13-32-38_huggingface_Qwen_Qwen2.5-7B-Instruct_featherless-ai"
RUN_DIR = RESULTS_ROOT / RUN_ID
CASES_PATH = RUN_DIR / "cases.jsonl"
SUMMARY_PATH = RUN_DIR / "summary.json"
MANIFEST_PATH = RUN_DIR / "manifest.json"


def load_json(path):
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)


def load_jsonl(path):
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


records = load_jsonl(CASES_PATH)
summary = load_json(SUMMARY_PATH)
manifest = load_json(MANIFEST_PATH)

print(f"Run: {manifest['run_id']}")
print(f"Cases: {len(records)}")
print(
    f"Generator: {manifest['generator_config']['provider']}/{manifest['generator_config']['model']}"
)
print(f"Judge: {manifest['judge_model']}")
print(f"Default threshold: {manifest['threshold']}")

Run: 2026-10-23_13-32-38_huggingface_Qwen_Qwen2.5-7B-Instruct_featherless-ai
Cases: 20
Generator: huggingface/Qwen/Qwen2.5-7B-Instruct:featherless-ai
Judge: gpt-4o-mini
Default threshold: 0.7


In [8]:
metric_summary = pd.DataFrame(
    [
        {
            "metric": name,
            "mean_score": values["mean_score"],
            "pass_rate": values["pass_rate"],
            "scored_cases": values["scored_cases"],
            "passed_cases": round(values["pass_rate"] * values["scored_cases"]),
        }
        for name, values in summary["metrics"].items()
    ]
)

run_summary = pd.DataFrame(
    {
        "value": {
            "attempted_cases": summary["cases"],
            "successful_cases": summary["successful_cases"],
            "error_cases": summary["error_cases"],
            "prompt_tokens": summary["prompt_tokens"],
            "completion_tokens": summary["completion_tokens"],
            "generation_latency_seconds": summary["generation_latency_seconds"],
        }
    }
)

display(run_summary)
display(metric_summary.style.format({"mean_score": "{:.3f}", "pass_rate": "{:.1%}"}))

,value
attempted_cases,20.000000
successful_cases,19.000000
error_cases,1.000000
prompt_tokens,17107.000000
completion_tokens,2072.000000
generation_latency_seconds,66.858272


,metric,mean_score,pass_rate,scored_cases,passed_cases
0,faithfulness,0.943,89.5%,19,17
1,answer_relevancy,0.908,89.5%,19,17
2,answer_correctness,0.829,89.5%,19,17


In [9]:
case_rows = []
metric_rows = []

for record in records:
    case = record["case"]
    case_id = case.get("source_qid", case.get("id"))
    case_rows.append(
        {
            "case_id": case_id,
            "status": record["status"],
            "doc_id": case.get("doc_id"),
            "query": case.get("query"),
            "error": record.get("error"),
        }
    )
    for metric_name, result in record.get("metrics", {}).items():
        metric_rows.append(
            {
                "case_id": case_id,
                "metric": metric_name,
                "score": result.get("score"),
                "threshold": result.get("threshold"),
                "passed": result.get("passed"),
                "judge_reason": result.get("reason"),
                "metric_error": result.get("error"),
                "evaluation_model": result.get("evaluation_model"),
            }
        )

cases_df = pd.DataFrame(case_rows)
metrics_df = pd.DataFrame(metric_rows)
error_cases_df = cases_df[cases_df["status"] != "success"].copy()
failed_metrics_df = metrics_df[
    (metrics_df["score"].isna())
    | (metrics_df["score"] < metrics_df["threshold"])
    | (~metrics_df["passed"].fillna(False))
].sort_values(["case_id", "metric"])

print(f"Pipeline/API error cases: {len(error_cases_df)}")
display(error_cases_df)
print(f"Below-threshold or errored metric results: {len(failed_metrics_df)}")
display(failed_metrics_df.style.format({"score": "{:.3f}", "threshold": "{:.3f}"}))

Pipeline/API error cases: 1


,case_id,status,doc_id,query,error
13,q014,error,MSFT_10-K_2026,Among Microsoft's executive officers as of Jul...,RetryError: RetryError[<Future at 0x157cc93245...


Below-threshold or errored metric results: 6


,case_id,metric,score,threshold,passed,judge_reason,metric_error,evaluation_model
6,q003,faithfulness,0.500,0.700,False,"The score is 0.50 because the actual output presents definitive revenue figures for 2024 and 2025, while the retrieval context refers to these figures as projections, indicating uncertainty.",None,gpt-4o-mini
10,q004,answer_relevancy,0.250,0.700,False,"The score is 0.25 because the irrelevant statements about India, the Middle East, and Africa being excluded from Apple's Europe reportable segment do not directly address the question regarding which geographic areas are included. These irrelevant details detract from the focus on the specific areas that are part of the segment, resulting in a lower relevancy score.",None,gpt-4o-mini
29,q010,answer_correctness,0.685,0.700,False,"The actual output correctly identifies the adverse effects of a weakening of foreign currencies on Apple's sales and earnings, aligning with the expected output. However, it omits the detail about the potential reduction in demand for Apple's products due to raising international pricing, which is a significant aspect of the expected response. Additionally, while the core information is maintained, the phrasing differs slightly, indicating a lack of full alignment with the expected output's structure.",None,gpt-4o-mini
45,q017,faithfulness,0.667,0.700,False,"The score is 0.67 because the actual output incorrectly presents the value of AI infrastructure in 2024 as a projected figure, while the retrieval context clearly states it as a definitive value of $5,152 million.",None,gpt-4o-mini
50,q018,answer_correctness,0.647,0.700,False,"The actual output correctly identifies the fiscal quarter in which Apple experiences higher net sales, aligning with the expected output. However, it omits the crucial detail regarding the reason for this pattern, which is attributed to seasonal holiday demand. This omission affects the completeness of the response, leading to a lower score despite the accurate identification of the quarter.",None,gpt-4o-mini
55,q020,answer_relevancy,0.000,0.700,False,"The score is 0.00 because the output fails to address the specific question about Microsoft's largest owned international properties, instead listing countries without relevant context. This lack of focus on the actual inquiry leads to a complete irrelevance in the response.",None,gpt-4o-mini


## Detailed failure inspection

The next view prints every failed metric together with the question, expected answer, actual answer, score, threshold, and the judge's complete reason.

In [10]:
records_by_id = {
    record["case"].get("source_qid", record["case"].get("id")): record for record in records
}

for row in failed_metrics_df.to_dict("records"):
    record = records_by_id[row["case_id"]]
    case = record["case"]
    display(
        Markdown(
            f"""### {row["case_id"]} — {row["metric"]}

**Score:** `{row["score"]}` &nbsp; **Threshold:** `{row["threshold"]}` &nbsp; **Passed:** `{row["passed"]}`

**Question**

{case.get("query")}

**Expected answer**

{case.get("expected_answer")}

**Actual answer**

{record.get("actual_output")}

**Judge reason**

{row["judge_reason"] or row["metric_error"] or "No reason recorded."}

---
"""
        )
    )

### q003 — faithfulness

**Score:** `0.5` &nbsp; **Threshold:** `0.7` &nbsp; **Passed:** `False`

**Question**

In Tesla's Revenues table, what was energy generation and storage segment revenue for the year ended December 31, 2025, and what were the dollar and percentage changes compared with the year ended December 31, 2024?

**Expected answer**

Energy generation and storage segment revenue was $12,771 million for the year ended December 31, 2025, up $2,685 million, or 27%, from $10,086 million for the year ended December 31, 2024.

**Actual answer**

$12,771 million in 2025, $10,086 million in 2024; $2,685 million increase (27%)

**Judge reason**

The score is 0.50 because the actual output presents definitive revenue figures for 2024 and 2025, while the retrieval context refers to these figures as projections, indicating uncertainty.

---


### q004 — answer_relevancy

**Score:** `0.25` &nbsp; **Threshold:** `0.7` &nbsp; **Passed:** `False`

**Question**

Which geographic areas are included in Apple's Europe reportable segment?

**Expected answer**

Apple's Europe segment includes European countries, as well as India, the Middle East and Africa.

**Actual answer**

European countries, as well as India, the Middle East and Africa

**Judge reason**

The score is 0.25 because the irrelevant statements about India, the Middle East, and Africa being excluded from Apple's Europe reportable segment do not directly address the question regarding which geographic areas are included. These irrelevant details detract from the focus on the specific areas that are part of the segment, resulting in a lower relevancy score.

---


### q010 — answer_correctness

**Score:** `0.6850634719832918` &nbsp; **Threshold:** `0.7` &nbsp; **Passed:** `False`

**Question**

According to Apple's risk factors, how does a weakening of foreign currencies relative to the U.S. dollar affect Apple, and what would be the effect if Apple decided not to raise international pricing to offset the U.S. dollar's strengthening?

**Expected answer**

A weakening of foreign currencies adversely affects the U.S. dollar value of Apple's foreign currency–denominated sales and earnings, and generally leads Apple to raise international pricing, potentially reducing demand for its products. If Apple decided not to raise international pricing, the U.S. dollar value of the gross margins it earns on foreign currency–denominated sales would be adversely affected.

**Actual answer**

A weakening of foreign currencies relative to the U.S. dollar adversely affects the U.S. dollar value of Apple’s foreign currency–denominated sales and earnings. If Apple decided not to raise international pricing to offset the U.S. dollar's strengthening, it would adversely affect the U.S. dollar value of the gross margins the Company earns on foreign currency–denominated sales.

**Judge reason**

The actual output correctly identifies the adverse effects of a weakening of foreign currencies on Apple's sales and earnings, aligning with the expected output. However, it omits the detail about the potential reduction in demand for Apple's products due to raising international pricing, which is a significant aspect of the expected response. Additionally, while the core information is maintained, the phrasing differs slightly, indicating a lack of full alignment with the expected output's structure.

---


### q017 — faithfulness

**Score:** `0.6666666666666666` &nbsp; **Threshold:** `0.7` &nbsp; **Passed:** `False`

**Question**

In Tesla's Note 6 property, plant and equipment table, what amounts were reported for AI infrastructure as of December 31, 2025 and December 31, 2024, and what was property, plant and equipment, net as of December 31, 2025 (in millions)?

**Expected answer**

AI infrastructure was 6,816 million as of December 31, 2025 and 5,152 million as of December 31, 2024. Property, plant and equipment, net was 40,643 million as of December 31, 2025.

**Actual answer**

AI infrastructure: $6,816 million (2025), $5,152 million (2024); Property, plant and equipment, net: $40,643 million (2025)

**Judge reason**

The score is 0.67 because the actual output incorrectly presents the value of AI infrastructure in 2024 as a projected figure, while the retrieval context clearly states it as a definitive value of $5,152 million.

---


### q018 — answer_correctness

**Score:** `0.6468183184417811` &nbsp; **Threshold:** `0.7` &nbsp; **Passed:** `False`

**Question**

In which fiscal quarter has Apple historically experienced higher net sales compared with its other quarters, and what reason does Apple give for this pattern?

**Expected answer**

Apple has historically experienced higher net sales in its first quarter compared with other quarters in its fiscal year, due in part to seasonal holiday demand.

**Actual answer**

Apple has historically experienced higher net sales in its first quarter compared with other quarters in its fiscal year.

**Judge reason**

The actual output correctly identifies the fiscal quarter in which Apple experiences higher net sales, aligning with the expected output. However, it omits the crucial detail regarding the reason for this pattern, which is attributed to seasonal holiday demand. This omission affects the completeness of the response, leading to a lower score despite the accurate identification of the quarter.

---


### q020 — answer_relevancy

**Score:** `0.0` &nbsp; **Threshold:** `0.7` &nbsp; **Passed:** `False`

**Question**

According to Microsoft's properties disclosure, in which locations are its largest owned international properties?

**Expected answer**

Microsoft's largest owned international properties include space in China, India, Ireland, the Netherlands, and Sweden.

**Actual answer**

China, India, Ireland, the Netherlands, and Sweden

**Judge reason**

The score is 0.00 because the output fails to address the specific question about Microsoft's largest owned international properties, instead listing countries without relevant context. This lack of focus on the actual inquiry leads to a complete irrelevance in the response.

---


## Inspect any individual case

Change `CASE_ID` to inspect all recorded metrics and reasons for one case, including cases that passed.

In [11]:
def inspect_case(case_id):
    record = records_by_id[case_id]
    case = record["case"]
    display(Markdown(f"## {case_id}: {case['query']}"))
    display(
        pd.DataFrame(
            {
                "value": {
                    "status": record["status"],
                    "document": case.get("doc_id"),
                    "expected_answer": case.get("expected_answer"),
                    "actual_output": record.get("actual_output"),
                    "pipeline_error": record.get("error"),
                }
            }
        )
    )
    case_metrics = metrics_df[metrics_df["case_id"] == case_id]
    if case_metrics.empty:
        print("No metric results were recorded for this case.")
    else:
        display(case_metrics.style.format({"score": "{:.3f}", "threshold": "{:.3f}"}))
    display(Markdown("### Ideal context"))
    for index, context in enumerate(case.get("ideal_context", []), start=1):
        display(Markdown(f"**Context {index}**\n\n```text\n{context}\n```"))
    return record


CASE_ID = "q003"
selected_record = inspect_case(CASE_ID)

## q003: In Tesla's Revenues table, what was energy generation and storage segment revenue for the year ended December 31, 2025, and what were the dollar and percentage changes compared with the year ended December 31, 2024?

,value
status,success
document,TSLA_10-K_2025
expected_answer,Energy generation and storage segment revenue ...
actual_output,"$12,771 million in 2025, $10,086 million in 20..."
pipeline_error,NaN


,case_id,metric,score,threshold,passed,judge_reason,metric_error,evaluation_model
6,q003,faithfulness,0.500,0.700,False,"The score is 0.50 because the actual output presents definitive revenue figures for 2024 and 2025, while the retrieval context refers to these figures as projections, indicating uncertainty.",None,gpt-4o-mini
7,q003,answer_relevancy,1.000,0.700,True,The score is 1.00 because the response directly addressed the question regarding Tesla's energy generation and storage segment revenue for the specified years without including any irrelevant statements.,None,gpt-4o-mini
8,q003,answer_correctness,0.797,0.700,True,"The actual output provides the correct revenue figures for both years and accurately states the dollar and percentage changes. However, it lacks the explicit phrasing 'up' before the increase, which is present in the expected output, slightly affecting the clarity of the relationship between the years. Overall, it captures the essential information well.",None,gpt-4o-mini


### Ideal context

**Context 1**

```text
**Revenues** (Dollars in millions)

| Line item | Year Ended December 31, 2025 | Year Ended December 31, 2024 | Year Ended December 31, 2023 | 2025 vs. 2024 Change $ | 2025 vs. 2024 Change % | 2024 vs. 2023 Change $ | 2024 vs. 2023 Change % |
|---|---|---|---|---|---|---|---|
| Automotive sales | 65,821 | 72,480 | 78,509 | -6,659 | -9% | -6,029 | -8% |
| Automotive regulatory credits | 1,993 | 2,763 | 1,790 | -770 | -28% | 973 | 54% |
| Automotive leasing | 1,712 | 1,827 | 2,120 | -115 | -6% | -293 | -14% |
| Total automotive revenues | 69,526 | 77,070 | 82,419 | -7,544 | -10% | -5,349 | -6% |
| Services and other | 12,530 | 10,534 | 8,319 | 1,996 | 19% | 2,215 | 27% |
| Total automotive & services and other segment revenue | 82,056 | 87,604 | 90,738 | -5,548 | -6% | -3,134 | -3% |
| Energy generation and storage segment revenue | 12,771 | 10,086 | 6,035 | 2,685 | 27% | 4,051 | 67% |
| Total revenues | 94,827 | 97,690 | 96,773 | -2,863 | -3% | 917 | 1% |
```

## All case scores

This matrix provides a quick comparison across all successfully scored cases. Missing values correspond to pipeline/API errors or metric errors.

In [12]:
score_matrix = metrics_df.pivot(index="case_id", columns="metric", values="score")
score_matrix = cases_df[["case_id", "status", "doc_id"]].merge(
    score_matrix, left_on="case_id", right_index=True, how="left"
)
metric_columns = [column for column in manifest["metrics"] if column in score_matrix.columns]
display(
    score_matrix.style.format(
        {column: "{:.3f}" for column in metric_columns}, na_rep="—"
    ).background_gradient(subset=metric_columns, cmap="RdYlGn", vmin=0, vmax=1)
)

ImportError: `Import matplotlib` failed. Styler.background_gradient requires matplotlib. Use pip or conda to install the matplotlib package.